In [16]:
import sys
!"{sys.executable}" -m pip install -q sentence-transformers scikit-learn numpy pandas matplotlib

In [17]:
# Импорт всех нужных библиотек
import json
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from sklearn.manifold import TSNE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Все библиотеки успешно импортированы")

In [18]:
# Находим папку с датасетом (три json-файла)
DATA_DIR = Path("dataset_v1.0")

with open(DATA_DIR / "code_corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)
with open(DATA_DIR / "eval_questions.json", encoding="utf-8") as f:
    questions = json.load(f)
with open(DATA_DIR / "categories.json", encoding="utf-8") as f:
    categories = json.load(f)["categories"]

print("Функций в корпусе:", len(corpus))
print("Тестовых вопросов:", len(questions))
print("Категорий:", len(categories))

In [19]:
# Выбираем две embedding-модели для сравнения
MODELS = {
    "MiniLM": "paraphrase-multilingual-MiniLM-L12-v2",
    "mpnet": "paraphrase-multilingual-mpnet-base-v2",
}

print("Будем сравнивать модели:")
for short_name, full_name in MODELS.items():
    print(f"  {short_name}  ->  {full_name}")

In [20]:
def build_text(item):
    return f"{item['function_name']}. {item['description']}\n{item['code']}"

doc_texts = [build_text(c) for c in corpus]
doc_ids = [c["id"] for c in corpus]

def search_top3(model, query, doc_embeddings):
    q_emb = model.encode(query, convert_to_tensor=True)
    sims = util.cos_sim(q_emb, doc_embeddings)[0].cpu().numpy()
    top_idx = np.argsort(-sims)[:3]
    return [doc_ids[i] for i in top_idx]

results = {}
for short_name, full_name in MODELS.items():
    print(f"Модель: {short_name}")
    print("скачивание и загрузка модели...")
    model = SentenceTransformer(full_name)

    print("кодирование 200 функций в векторы...")
    doc_emb = model.encode(doc_texts, convert_to_tensor=True, show_progress_bar=False)

    print("поиск топ-3 по каждому из 25 вопросов...")
    preds = {}
    for q in questions:
        preds[q["question_id"]] = search_top3(model, q["query"], doc_emb)

    results[short_name] = {"doc_emb": doc_emb.cpu().numpy(), "preds": preds}
    print("готово.\n")

print("Все модели отработали.")

In [21]:
def precision_at_3(preds):
    hits = 0
    for q in questions:
        if q["correct_chunk_id"] in preds[q["question_id"]]:
            hits += 1
    return hits / len(questions)

for short_name in results:
    results[short_name]["p_at_3"] = precision_at_3(results[short_name]["preds"])
    print(f"{short_name:8s}  Precision@3 = {results[short_name]['p_at_3']:.3f}")

In [22]:
# Собираем результаты в таблицу и сохраняем в файл
comparison = pd.DataFrame(
    [{"Модель": name, "Precision@3": round(results[name]["p_at_3"], 3)}
     for name in results]
).sort_values("Precision@3", ascending=False).reset_index(drop=True)

comparison.to_csv("comparison_table.csv", index=False, encoding="utf-8-sig")

# выводим таблицу на экран
from IPython.display import display
display(comparison)
print(comparison.to_string(index=False))

In [23]:
best_model = comparison.iloc[0]["Модель"]
print("Лучшая модель:", best_model)

emb = results[best_model]["doc_emb"]
perplexity = min(30, len(corpus) - 1)
coords = TSNE(n_components=2, perplexity=perplexity,
              init="pca", random_state=RANDOM_STATE).fit_transform(emb)

cat_info = {c["key"]: c for c in categories}

plt.figure(figsize=(10, 8))
for key, info in cat_info.items():
    idx = [i for i, c in enumerate(corpus) if c["category"] == key]
    plt.scatter(coords[idx, 0], coords[idx, 1],
                c=info["color"], label=info["label"], s=35, alpha=0.8)

plt.title(f"t-SNE эмбеддингов корпуса (модель: {best_model})")
plt.xlabel("компонента 1")
plt.ylabel("компонента 2")
plt.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.savefig("tsne_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("График сохранён в tsne_plot.png")